# 3.01 In-Memory · FAISS Vector Store

외부 서비스가 필요 없는 **완전 로컬** 벡터 스토어 두 가지를 다룬다.
- `InMemoryVectorStore` (langchain-core) — 프로세스 메모리에만 존재. 단위 테스트·노트북 실습용.
- `FAISS` (langchain-community) — Meta FAIR 제작 ANN 라이브러리. 디스크 직렬화 지원.

## 학습 목표

- `.from_texts` / `.from_documents` 팩토리로 초기화하고 `similarity_search(k)`를 호출한다
- 메타데이터 필터(dict)로 결과를 좁힌다
- `similarity_search_with_score` · `max_marginal_relevance_search`의 차이를 이해한다
- `FAISS.save_local` / `load_local`로 인덱스를 영속화한다

## 언제 쓰나

- 프로토타입·단위 테스트에서 외부 DB 없이 RAG를 실험
- 수만 문서 수준의 **로컬 배치**를 디스크에 저장해 CI에서 재사용
- 네트워크 서비스 도입 전에 임베딩 품질을 검증

## 3.01.1 환경 설정

`InMemoryVectorStore`는 `langchain-core` 기본, `FAISS`는 `faiss-cpu` 또는 `faiss-gpu`가 필요하다. 임베딩만 OpenAI를 쓰므로 `OPENAI_API_KEY`만 있으면 된다.

In [ ]:
# !pip install -U langchain langchain-core langchain-openai langchain-community faiss-cpu

import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요 (임베딩용)"

## 3.01.2 기본 사용법 — `InMemoryVectorStore`

샘플 한국어 문서 5개로 시작한다. 모든 문서는 `source`, `topic` 메타데이터를 가진다.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="LangChain은 LLM 애플리케이션 구축을 위한 프레임워크이다.", metadata={"source": "docs", "topic": "intro"}),
    Document(page_content="LangGraph는 상태가 있는 멀티 에이전트 워크플로우를 만드는 라이브러리이다.", metadata={"source": "docs", "topic": "graph"}),
    Document(page_content="FAISS는 Meta에서 만든 근사 최근접 이웃 검색 라이브러리이다.", metadata={"source": "wiki", "topic": "faiss"}),
    Document(page_content="벡터 스토어는 임베딩을 저장하고 유사도 검색을 제공한다.", metadata={"source": "docs", "topic": "intro"}),
    Document(page_content="RAG는 검색 증강 생성으로, 외부 지식을 LLM 응답에 붙인다.", metadata={"source": "blog", "topic": "rag"}),
]

mem_store = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
hits = mem_store.similarity_search("벡터 검색이란 무엇인가", k=3)
for d in hits:
    print("-", d.page_content, "|", d.metadata)

## 3.01.3 메타데이터 필터링

`InMemoryVectorStore`는 callable 필터(`filter=lambda doc: ...`)를 지원한다. 실제 프로덕션 백엔드(Pinecone/Qdrant)는 DSL dict를 쓰지만, 로컬 메모리 구현은 **문서 객체를 직접 평가**하는 파이썬 콜러블이다.

In [ ]:
# source가 'docs'인 문서만 대상
only_docs = mem_store.similarity_search(
    "LangGraph 워크플로우",
    k=3,
    filter=lambda d: d.metadata.get("source") == "docs",
)
for d in only_docs:
    print("-", d.metadata["topic"], "|", d.page_content[:40])

## 3.01.4 점수 포함 · MMR

- `similarity_search_with_score(query, k)` — `(Document, float)` 튜플 반환. 점수 의미는 백엔드마다 다르다 (InMemory·FAISS는 코사인 거리 또는 L2 거리).
- `max_marginal_relevance_search(query, k, fetch_k, lambda_mult)` — **다양성**을 고려해 중복을 줄인다. `lambda_mult=1.0`이면 순수 유사도, `0.0`이면 최대 다양성.

In [ ]:
print("--- with_score ---")
for doc, score in mem_store.similarity_search_with_score("RAG", k=3):
    print(f"{score:.4f}  {doc.page_content[:50]}")

print("\n--- MMR (다양성 강조) ---")
mmr_hits = mem_store.max_marginal_relevance_search("LangChain 생태계", k=3, fetch_k=5, lambda_mult=0.3)
for d in mmr_hits:
    print("-", d.metadata["topic"], "|", d.page_content[:50])

## 3.01.5 FAISS — 디스크 영속화

`FAISS.save_local(folder_path)` 로 `index.faiss` + `index.pkl` 두 파일을 남긴다. 재시작 후 `FAISS.load_local(folder_path, embeddings, allow_dangerous_deserialization=True)` 로 복원.

`allow_dangerous_deserialization=True`는 파이썬 직렬화 포맷을 사용한 문서 메타 복원을 허용한다 — **신뢰할 수 없는 외부 파일에는 절대 쓰지 않는다**.

In [ ]:
from langchain_community.vectorstores import FAISS
import shutil, tempfile, os as _os

faiss_store = FAISS.from_documents(docs, embedding=embeddings)

tmp_dir = tempfile.mkdtemp(prefix="faiss_")
faiss_store.save_local(tmp_dir)
print("saved:", sorted(_os.listdir(tmp_dir)))

reloaded = FAISS.load_local(tmp_dir, embeddings=embeddings, allow_dangerous_deserialization=True)
print("reloaded count:", len(reloaded.index_to_docstore_id))
for d, s in reloaded.similarity_search_with_score("임베딩 저장소", k=2):
    print(f"{s:.4f}  {d.page_content[:50]}")

shutil.rmtree(tmp_dir)

## 3.01.6 FAISS — 증분 추가 · 삭제

`add_documents(docs, ids=...)` 로 기존 인덱스에 문서를 추가할 수 있다. 반환값은 저장된 ID 리스트.
`delete(ids=...)` 로 ID 단위 제거.

In [ ]:
new_ids = faiss_store.add_documents([
    Document(page_content="HNSW는 FAISS가 지원하는 계층적 근사 최근접 이웃 알고리즘이다.", metadata={"source": "wiki", "topic": "faiss"}),
])
print("added ids:", new_ids)
print("total:", len(faiss_store.index_to_docstore_id))

faiss_store.delete(ids=new_ids)
print("after delete:", len(faiss_store.index_to_docstore_id))

## 3.01.7 Retriever 변환

모든 `VectorStore`는 `as_retriever(search_type=..., search_kwargs=...)` 로 LangChain `Runnable` 체인에 꽂을 수 있다.
`search_type`은 `"similarity"` · `"mmr"` · `"similarity_score_threshold"` 중 선택.

In [ ]:
retriever = faiss_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5, "lambda_mult": 0.5},
)
for d in retriever.invoke("LangChain 프레임워크"):
    print("-", d.page_content[:50])

## 체크리스트

- [ ] `InMemoryVectorStore`는 프로세스 재시작 시 사라진다 — 영속화 필요하면 FAISS·Chroma로 교체
- [ ] `FAISS.load_local`에 `allow_dangerous_deserialization=True`는 **자기 소유 파일**에만 사용
- [ ] 대량 삽입 시 `from_documents`보다 `add_documents`로 배치 단위 추가가 메모리 효율적
- [ ] 유사도 점수 스케일은 백엔드·distance_strategy마다 다름 → 앱에서 임계값 하드코딩 금지

## 다음

- `02_chroma.ipynb` — `persist_directory` 기반 로컬 DB, 메타데이터 필터 DSL
- `05_retrievers/` — BM25 · multi-vector · parent-document retriever

## 참고

- LangChain Vector Stores: https://python.langchain.com/docs/integrations/vectorstores/
- FAISS wiki: https://github.com/facebookresearch/faiss/wiki